In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import pickle

# 1. Load your dataset
# Update the filename to match your local file path
df = pd.read_csv('IPL.csv')

# 2. Basic Cleaning
# Filter out matches with no result or abandoned
df = df[df['win_outcome'] != 'no result']
df = df[df['win_outcome'] != 'abandoned']

# 3. Create 'Match Winner' Binary Column
# 1 if Batting Team won, 0 if Bowling Team won
df['winner_is_batting_team'] = np.where(df['batting_team'] == df['match_won_by'], 1, 0)

# 4. Extract 2nd Innings Only
# Win probability is most dynamic during the chase
chase_df = df[df['innings'] == 2].copy()

# 5. Feature Engineering (The most important part)
# Calculate current score and wickets at every ball
chase_df['current_score'] = chase_df.groupby('match_id')['runs_total'].cumsum()

# To get 'wickets_lost', we check if 'striker_out' is not null
chase_df['is_wicket'] = chase_df['striker_out'].notnull().astype(int)
chase_df['wickets_lost'] = chase_df.groupby('match_id')['is_wicket'].cumsum()

# Target and Remaining resources
chase_df['runs_left'] = chase_df['runs_target'] - chase_df['current_score']
chase_df['balls_left'] = 120 - ((chase_df['over'] * 6) + chase_df['ball'])
chase_df['wickets_remaining'] = 10 - chase_df['wickets_lost']

# Calculate Rates
# (Adding 0.001 to avoid division by zero errors)
chase_df['crr'] = (chase_df['current_score'] * 6) / (120 - chase_df['balls_left'])
chase_df['rrr'] = (chase_df['runs_left'] * 6) / chase_df['balls_left'].replace(0, 1)

# 6. Final Feature Selection
final_df = chase_df[[
    'batting_team', 'bowling_team', 'city', 'runs_left', 
    'balls_left', 'wickets_remaining', 'runs_target', 'rrr', 'crr', 'winner_is_batting_team'
]]

# Drop any remaining rows with missing values
final_df.dropna(inplace=True)

# 7. Encoding Categorical Data
# We use One-Hot Encoding so the model treats teams as unique identities
final_df = pd.get_dummies(final_df, columns=['batting_team', 'bowling_team', 'city'])

# 8. Train-Test Split
X = final_df.drop('winner_is_batting_team', axis=1)
y = final_df['winner_is_batting_team']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 9. Model Training
# We use Random Forest because it handles the complex relationships in cricket well
model = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42)
model.fit(X_train, y_train)

# 10. Save the Model and the Column Structure
# Your Streamlit app MUST know the order of columns to work
model_data = {
    'model': model,
    'columns': X.columns.tolist()
}

with open('ipl_model_v4.pkl', 'wb') as f:
    pickle.dump(model_data, f)

print(f"Success! Model trained on {len(final_df)} balls.")
print(f"Accuracy on Test Data: {model.score(X_test, y_test):.2%}")

/var/folders/_g/hfrndkl15m31tqxd1_8ysqt80000gn/T/ipykernel_64545/4018264021.py:10: DtypeWarning: Columns (28,29,30,31,43,46,47,48,51) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('IPL.csv')
/var/folders/_g/hfrndkl15m31tqxd1_8ysqt80000gn/T/ipykernel_64545/4018264021.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df.dropna(inplace=True)


Success! Model trained on 133903 balls.
Accuracy on Test Data: 96.31%
